# Pelatihan Model MobileNetV2 untuk Deteksi Covid-19

Notebook ini dibuat untuk melatih model deteksi Covid-19 menggunakan arsitektur **MobileNetV2**. Dataset yang digunakan berasal dari *ieee8023/covid-chestxray-dataset*.

**Langkah-langkah yang dilakukan:**
1. Mengunduh dataset dari GitHub (jika dijalankan di Google Colab).
2. Membaca file `metadata.csv` untuk memisahkan gambar paru-paru Covid-19 dengan yang Normal/Pneumonia biasa.
3. Membangun model *Transfer Learning* berbasis MobileNetV2.
4. Melatih model dan menyimpannya sebagai `covid_mobilenet_model.h5`.

In [ ]:
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

## 1. Persiapan Dataset
Jika Anda menjalankan ini di Colab, kita akan *clone* repository dataset-nya langsung.

In [ ]:
# Clone dataset jika belum ada
if not os.path.exists('covid-chestxray-dataset'):
    !git clone https://github.com/ieee8023/covid-chestxray-dataset.git

### Membaca `metadata.csv` dan Menyusun Direktori
Kita akan membuat folder `dataset/train/Covid` dan `dataset/train/Normal` lalu memindahkan gambar dari folder `images/` sesuai dengan label yang ada di `metadata.csv`.

In [ ]:
metadata_path = 'covid-chestxray-dataset/metadata.csv'
images_dir = 'covid-chestxray-dataset/images'

base_dir = 'dataset/train'
covid_dir = os.path.join(base_dir, 'Covid')
normal_dir = os.path.join(base_dir, 'Normal')

os.makedirs(covid_dir, exist_ok=True)
os.makedirs(normal_dir, exist_ok=True)

# Membaca CSV
df = pd.read_csv(metadata_path)

covid_count = 0
normal_count = 0

for index, row in df.iterrows():
    filename = row['filename']
    finding = row['finding']
    # Dataset ini berisi pandangan x-ray yang berbeda, kita ambil yang frontal (PA/AP)
    view = row['view']
    
    img_path = os.path.join(images_dir, filename)
    
    if os.path.exists(img_path) and view in ['PA', 'AP', 'AP Supine']:
        if 'COVID-19' in finding:
            shutil.copy(img_path, os.path.join(covid_dir, filename))
            covid_count += 1
        # Beberapa data mungkin Normal atau jenis pneumonia lain yang bukan Covid-19
        elif 'COVID-19' not in finding:
            shutil.copy(img_path, os.path.join(normal_dir, filename))
            normal_count += 1

print(f"Total gambar Covid-19: {covid_count}")
print(f"Total gambar Non-Covid/Normal: {normal_count}")

## 2. Preprocessing & Data Augmentation
Kita menggunakan `ImageDataGenerator` yang khusus untuk MobileNetV2 (`tf.keras.applications.mobilenet_v2.preprocess_input`).

In [ ]:
IMG_SIZE = 224 # Ukuran standar MobileNetV2
BATCH_SIZE = 32

# Preprocessing khusus MobileNetV2 (skala pixel ke rentang [-1, 1])
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 20% data digunakan untuk validasi/testing
)

train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

## 3. Membangun Model MobileNetV2

In [ ]:
# Load base model MobileNetV2 (tanpa fully-connected layer di atasnya)
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')

# Bekukan (freeze) base model agar bobotnya tidak berubah saat awal training
base_model.trainable = False

# Tambahkan layer klasifikasi khusus
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x) # Sigmoid untuk klasifikasi biner

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()

## 4. Melatih Model (Training)

In [ ]:
EPOCHS = 10

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

## 5. Visualisasi Hasil Training

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss')
plt.show()

## 6. Menyimpan Model
Simpan model utuh (beserta arsitektur dan bobotnya) menjadi file `.h5`. File ini yang nanti akan ditaruh di dalam folder project web Flask Anda.

In [ ]:
model.save('covid_mobilenet_model.h5')
print("Model berhasil disimpan sebagai covid_mobilenet_model.h5")